# ST-Transformer

这里，我们先回顾 Transformer 的实现，然后重点聚焦于 ST-Transformer。

## 1. Transformer

Transformer 的架构和数据流动如图所示：

<div style="display:flex;flex-wrap:wrap;gap:10px;justify-content:center;">
  <img src="assets/transformer_architecture.webp" style="flex:1;min-width:280px;max-width:700px;height:auto;">
  <img src="assets/transformer_data_flow.png" style="flex:1;min-width:280px;max-width:700px;height:auto;">
</div>

在很久之前（二六年三月），我其实已经手写了一个 Transformer 架构，我经过部分修改后，移植到了这个系列之中，可以参考那份 Notebook。

## 2. ST-Transformer Overview

ST-Transformer 专门用于解决视频序列的痛点。对于视频序列 $$ X\in \mathbb R^{B\times T\times H\times W\times d} $$，若按一般的方法装入 Transformer，我们需要把图像时间全部展平，变成 $(B, THW, d)$ 的序列，这里的计算量级达 $B(THW)^2$，非常大。

为了解决这个痛点，ST-Transformer 分两次处理序列，一次是按单张图像一个一个处理，将原来的图像转换为 $(BT, HW, d)$；另一次是按单个图像上某点在时间序列中的变化来处理，即将原来的图像转换为 $(BHW, T, d)$。这样一共的计算量量级为 $BT(HW)^2 + BHWT^2$，更加友好。

其实 ST-Transformer 似乎没有我想象得那么复杂。整体架构差不多是这个样子：

<div align="center">
  <img src="assets/st_transformer_architecture.png" width="800">
</div>

## 3. Implementations

我们将目标缩减为仅构造一个 encoder-only 的模型。这里我们先构造一个 `STBlock`（对应我们之前文档的命名，应该叫 `EncoderLayer`）。

In [2]:
import torch
import torch.nn as nn

In [3]:
class STBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4):
        # mlp_ratio 控制隐藏层相对于输入维度大小的比值

        super().__init__()

        self.spatial_norm = nn.LayerNorm(dim)
        self.spatial_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)

        self.temporal_norm = nn.LayerNorm(dim)
        self.temporal_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)

        self.norm_ff = nn.LayerNorm(dim)

        self.ff = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        # x: (B, T, H, W, D)

        B, T, H, W, D = x.shape

        # --- spatial ---
        xs = self.spatial_norm(x)
        xs = xs.reshape(B * T, H * W, D)
        xs, _ = self.spatial_attn(xs, xs, xs, need_weights=False) # self attention
        xs = xs.reshape(B, T, H, W, D)
        x = x + xs

        # --- temporal ---
        xt = self.temporal_norm(x)
        xt = xt.permute(0, 2, 3, 1, 4).reshape(B * H * W, T, D)
        xt, _ = self.temporal_attn(xt, xt, xt, need_weights=False)
        xt = xt.reshape(B, H, W, T, D).permute(0, 3, 1, 2, 4)
        x = x + xt

        # --- FFN ---
        x = x + self.ff(self.norm_ff(x))

        return x

这里我们对部分代码细节进行说明。

首先，我们可以看到，`x` 是逐次通过两层自注意力的计算的。

其次，在原初的 Transformer 中，是先计算 `x + F(x)`，然后进行 `layer_norm`。但在现代工业实践中，Pre-LN 往往更常见，具有更强的稳定性，故我们代码中实现的是 `x = x + self.ff(self.norm_ff(x))`（包括前面的 `spatial_norm` 与 `temporal_norm` 亦是如此）。

有了刚刚的基础，我们把 Block 叠起来，然后补上前后的一些处理流程，就构成了目标 ST-Transformer。

In [4]:
class STTransformer(nn.Module):
    def __init__(
        self,
        in_channels=3,
        dim=256,
        image_size=64,
        patch_size=8,
        num_frames=16,
        num_blocks=8,
        num_heads=8,
        mlp_ratio=4,
    ):
        super().__init__()

        self.dim = dim
        self.patch_size = patch_size

        assert image_size % patch_size == 0
        H = image_size // patch_size
        W = image_size // patch_size

        self.H = H
        self.W = W
        self.num_frames = num_frames

        # patch embedding
        # Compress image to prevent an explosion in computational complexity.
        self.patch_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)

        # positional embeddings (learnable)
        self.temporal_pos = nn.Parameter(torch.randn(1, num_frames, 1, 1, dim) * 0.02)
        self.spatial_pos = nn.Parameter(torch.randn(1, 1, H, W, dim) * 0.02)

        # ST-Transformer blocks
        self.blocks = nn.ModuleList([
            STBlock(dim=dim, num_heads=num_heads, mlp_ratio=mlp_ratio)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x_input):
        # (B, T, C, H_, W_)

        B, T, C, H_, W_ = x_input.shape
        assert T <= self.num_frames
        x_input = x_input.reshape(B * T, C, H_, W_)

        # (B * T, D, H, W)
        x = self.patch_embed(x_input) 
        _, D, H, W = x.shape

        # (B, T, H, W, D)
        x = x.reshape(B, T, D, H, W).permute(0, 1, 3, 4, 2)
        
        x = (x + self.temporal_pos[:, :T] + self.spatial_pos[:, :, :H, :W])

        for block in self.blocks:
            x = block(x)
        x = self.norm(x) # (B, T, H, W, D)

        return x

注意，里面的位置编码是可学习的参数，与 Transformer 原始论文不同。

现在，我们构造一组伪数据，确保定义无误（这里没办法拿具体数据训练）。

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [13]:
B, T, C, H_, W_ = 16, 16, 3, 64, 64
X_test = torch.randn(B, T, C, H_, W_).to(device)
model = STTransformer().to(device)
y = model(X_test)
print(X_test.shape, y.shape)
print(y[0][0][0:2][0:2][0:2])

torch.Size([16, 16, 3, 64, 64]) torch.Size([16, 16, 8, 8, 256])
tensor([[[-0.4330,  0.1561,  1.5146,  ..., -0.9121, -0.8329,  1.0438],
         [ 0.4546, -0.4088,  0.0071,  ..., -1.2571,  1.4060, -0.6559],
         [ 0.4583, -0.2285,  0.7743,  ..., -1.0195, -0.1593, -0.7767],
         ...,
         [-0.8403, -0.2309, -0.0644,  ..., -1.2055,  0.2205,  0.4189],
         [ 1.6867,  0.4185,  1.5760,  ..., -1.6253,  0.1526,  1.3395],
         [ 0.0593, -0.6923,  0.3369,  ..., -1.2857, -0.3857,  0.4390]],

        [[ 1.1039,  0.3759, -0.8030,  ..., -3.0827, -1.2405,  0.8461],
         [ 0.1973, -1.0103,  0.5241,  ..., -1.6450, -0.2329, -0.5585],
         [-0.5876,  0.2477,  0.5494,  ..., -0.1468, -1.1260, -0.1110],
         ...,
         [-1.5472, -1.5540, -0.5234,  ..., -0.4630,  0.4615,  0.2836],
         [ 0.3405, -0.1600,  0.4402,  ..., -1.9764, -0.1811,  0.4095],
         [ 0.9131,  0.6470,  0.9213,  ..., -2.5351, -0.2476,  1.0980]]],
       device='cuda:0', grad_fn=<SliceBackward0>)


数据没问题，那这部分就差不多结束了。